In [2]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv("citibike_tripdata_september_2025_cleaned.csv")

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 116055 entries, 0 to 116054
Data columns (total 25 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   ride_id             116055 non-null  str    
 1   rideable_type       116055 non-null  str    
 2   started_at          116055 non-null  str    
 3   ended_at            116055 non-null  str    
 4   start_station_name  116055 non-null  str    
 5   start_station_id    116055 non-null  str    
 6   end_station_name    115694 non-null  str    
 7   end_station_id      115568 non-null  str    
 8   start_lat           116055 non-null  float64
 9   start_lng           116055 non-null  float64
 10  end_lat             115568 non-null  float64
 11  end_lng             115568 non-null  float64
 12  member_casual       116055 non-null  str    
 13  duration_min        116055 non-null  float64
 14  duration_outlier    116055 non-null  bool   
 15  ride_incomplete     116055 non-null  bool   


In [12]:
#Checking for Null Values
df.isnull().sum()

ride_id                 0
rideable_type           0
started_at              0
ended_at                0
start_station_name      0
start_station_id        0
end_station_name      361
end_station_id        487
start_lat               0
start_lng               0
end_lat               487
end_lng               487
member_casual           0
duration_min            0
duration_outlier        0
ride_incomplete         0
cross_river_trip        0
date                    0
hour_of_day             0
day_of_week             0
day_name                0
is_weekday              0
distance_km           487
speed_kmh             487
round_trip              0
dtype: int64

In [14]:
#Converting the Stating and Ending Time to DateTime Data Type
df["started_at"] = pd.to_datetime(df["started_at"])
df["ended_at"] = pd.to_datetime(df["ended_at"])
df[["started_at","ended_at"]].dtypes

started_at    datetime64[us]
ended_at      datetime64[us]
dtype: object

In [16]:

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
MEMBER_COLOR = "#2196F3"   # blue  — members
CASUAL_COLOR = "#FF9800"   # amber — casuals

In [ ]:
# Chart 1 — Member vs Casual Trip Duration Distribution (0–60 min, outliers >240 min excluded)
# Business question: How differently do members and casuals use the network?
# Excluding duration outliers (>60 min) for clean visualization

fig, ax = plt.subplots(figsize=(10, 5))
 
clean = df[(df["duration_outlier"] == False) & (df["ride_incomplete"] == False)]
 
for rider, color, label in [
    ("member", MEMBER_COLOR, f"Member (n={clean[clean.member_casual=='member'].shape[0]:,})"),
    ("casual", CASUAL_COLOR, f"Casual (n={clean[clean.member_casual=='casual'].shape[0]:,})")
]:
    subset = clean[clean["member_casual"] == rider]["duration_min"]
    ax.hist(subset, bins=60, range=(0, 60), alpha=0.6,
            color=color, label=label, density=True)
 
ax.set_xlabel("Trip Duration (minutes)")
ax.set_ylabel("Density")
ax.set_title("Chart 1 — Trip Duration Distribution: Member vs Casual\n"
             "(Outliers >240 min excluded, trips 0–60 min shown)")
ax.legend()
 
member_med = clean[clean.member_casual == "member"]["duration_min"].median()
casual_med = clean[clean.member_casual == "casual"]["duration_min"].median()
ax.axvline(member_med, color=MEMBER_COLOR, linestyle="--", linewidth=1.5,
           label=f"Member median: {member_med:.1f} min")
ax.axvline(casual_med, color=CASUAL_COLOR, linestyle="--", linewidth=1.5,
           label=f"Casual median: {casual_med:.1f} min")
ax.legend()
 
ax.text(0.98, 0.95,
        f"Member median: {member_med:.1f} min\nCasual median: {casual_med:.1f} min",
        transform=ax.transAxes, ha="right", va="top",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8), fontsize=10)
 
plt.tight_layout()
plt.savefig("Chart_1_duration_distribution.png", dpi=500)
plt.close()
print("Chart 1 saved successfully.")

In [ ]:
# Chart 2 — HOURLY DEMAND HEATMAP (hour × day-of-week, total rides)
# Business question: When is the system under most stress?

day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday",
             "Saturday", "Sunday"]
 
heatmap_data = (
    df.groupby(["day_name", "hour_of_day"])
    .size()
    .reset_index(name="rides")
    .pivot(index="day_name", columns="hour_of_day", values="rides")
    .reindex(day_order)
)
 
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    heatmap_data,
    ax=ax,
    cmap="YlOrRd",
    linewidths=0.3,
    linecolor="white",
    annot=False,
    fmt="d",
    cbar_kws={"label": "Ride Count"}
)
ax.set_title("Chart 2 — Ride Demand Heatmap: Hour of Day × Day of Week\n"
             "(All rides, September 2025)")
ax.set_xlabel("Hour of Day (0 = midnight)")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("Chart_2_demand_heatmap.png", dpi=500)
plt.close()
print("Chart 2 saved successfully.")

In [ ]:
# Chart 3 — MEMBER VS CASUAL HOURLY DEMAND OVERLAY
# Business question: Are peaks structurally different between rider types?
# This is the chart that justifies "rebalance before 7am not during peak"
hourly = (
    df.groupby(["hour_of_day", "member_casual"])
    .size()
    .reset_index(name="rides")
)
 
fig, ax = plt.subplots(figsize=(12, 5))
 
for rider, color in [("member", MEMBER_COLOR), ("casual", CASUAL_COLOR)]:
    subset = hourly[hourly["member_casual"] == rider]
    ax.plot(subset["hour_of_day"], subset["rides"],
            color=color, linewidth=2.5, marker="o", markersize=4,
            label=rider.capitalize())
    ax.fill_between(subset["hour_of_day"], subset["rides"],
                    alpha=0.15, color=color)
 
ax.set_xticks(range(0, 24))
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Total Rides (Sept 2025)")
ax.set_title("Chart 3 — Hourly Demand: Member vs Casual\n"
             "(Sharp dual peaks for members; casual demand spread across the day)")
ax.legend()
ax.axvspan(7, 9, alpha=0.08, color="red", label="AM Peak")
ax.axvspan(16, 18, alpha=0.08, color="orange", label="PM Peak")
ax.legend()
 
plt.tight_layout()
plt.savefig("Chart_3_hourly_member_vs_casual.png", dpi=500)
plt.close()
print("Chart 3 saved successfully.")

In [ ]:

# Chart 4 — TRIP DISTANCE DISTRIBUTION (completed, non-round-trip rides only)
# Business question: What kind of trips dominate — short hops or commutes?
# ---------------------------------------------------------------------------
dist_clean = df[
    (df["ride_incomplete"] == False) &
    (df["round_trip"] == False) &
    (df["distance_km"].notna()) &
    (df["distance_km"] > 0)
]
 
fig, ax = plt.subplots(figsize=(10, 5))
 
for rider, color in [("member", MEMBER_COLOR), ("casual", CASUAL_COLOR)]:
    subset = dist_clean[dist_clean["member_casual"] == rider]["distance_km"]
    ax.hist(subset, bins=50, range=(0, 6), alpha=0.6,
            color=color, label=f"{rider.capitalize()} (n={len(subset):,})",
            density=True)
 
ax.set_xlabel("Trip Distance — Haversine straight-line (km)")
ax.set_ylabel("Density")
ax.set_title("Chart 4 — Trip Distance Distribution: Member vs Casual\n"
             "(Completed non-round trips only; 0–6 km range shown)")
 
member_med_d = dist_clean[dist_clean.member_casual == "member"]["distance_km"].median()
casual_med_d = dist_clean[dist_clean.member_casual == "casual"]["distance_km"].median()
 
ax.text(0.98, 0.95,
        f"Member median: {member_med_d:.2f} km\nCasual median: {casual_med_d:.2f} km",
        transform=ax.transAxes, ha="right", va="top",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8), fontsize=10)
ax.legend()
 
plt.tight_layout()
plt.savefig("Chart_4_distance_distribution.png", dpi=500)
plt.close()
print("Chart 4 saved successfully.")

In [29]:
# Chart 5 — Top 15 Stations by Total Outbound Rides
# Business question: Which stations are the highest-volume origins?

top_stations = (
    df[df["ride_incomplete"] == False]
    .groupby(["start_station_id","start_station_name"])
    .size()
    .sort_values(ascending=False)
    .head(15)
    .reset_index(name="rides")
)
 
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    top_stations["start_station_name"][::-1],
    top_stations["rides"][::-1],
    color=MEMBER_COLOR, alpha=0.85
)
 
for bar, val in zip(bars, top_stations["rides"][::-1]):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height() / 2,
            f"{val:,}", va="center", fontsize=9)
 
ax.set_xlabel("Total Outbound Rides (September 2025)")
ax.set_title("Chart 5 — Top 15 Stations by Outbound Ride Volume")
ax.set_xlim(0, top_stations["rides"].max() * 1.12)
plt.tight_layout()
plt.savefig("Chart_5_top_stations_outbound.png", dpi=500)
plt.close()
print("Chart 5 saved successfully.")

Chart 5 saved successfully.


In [30]:
# Chart 6 — NET FLOW: CHRONIC SURPLUS VS DEFICIT STATIONS
# Business question: Which stations need rebalancing trucks most urgently?
# Uses Q2 logic: avg daily net flow across September
# ---------------------------------------------------------------------------
rides_out = (
    df[df["ride_incomplete"] == False]
    .groupby(["start_station_name", "date"])
    .size()
    .reset_index(name="outbound")
    .rename(columns={"start_station_name": "station"})
)
 
rides_in = (
    df[(df["ride_incomplete"] == False) & (df["cross_river_trip"] == False)]
    .groupby(["end_station_name", "date"])
    .size()
    .reset_index(name="inbound")
    .rename(columns={"end_station_name": "station"})
)
 
daily = pd.merge(rides_out, rides_in, on=["station", "date"], how="outer").fillna(0)
daily["net_flow"] = daily["inbound"] - daily["outbound"]
 
avg_flow = (
    daily.groupby("station")["net_flow"]
    .mean()
    .reset_index(name="avg_daily_net_flow")
    .sort_values("avg_daily_net_flow")
)
 
# Show top 10 deficit + top 10 surplus only
plot_df = pd.concat([avg_flow.head(10), avg_flow.tail(10)]).drop_duplicates()
plot_df = plot_df.sort_values("avg_daily_net_flow")
colors = [CASUAL_COLOR if v < 0 else MEMBER_COLOR
          for v in plot_df["avg_daily_net_flow"]]
 
fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(plot_df["station"], plot_df["avg_daily_net_flow"],
               color=colors, alpha=0.85)
 
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
 
for bar, val in zip(bars, plot_df["avg_daily_net_flow"]):
    xpos = bar.get_width() + (0.1 if val >= 0 else -0.1)
    ha = "left" if val >= 0 else "right"
    ax.text(xpos, bar.get_y() + bar.get_height() / 2,
            f"{val:+.1f}", va="center", ha=ha, fontsize=8.5)
 
ax.set_xlabel("Average Daily Net Flow (Inbound − Outbound)")
ax.set_title("Chart 6 — Station Net Flow: Top 10 Deficit vs Top 10 Surplus\n"
             "(Amber = chronic deficit / bikes drain out; Blue = chronic surplus / bikes pile up)")
plt.tight_layout()
plt.savefig("Chart_6_net_flow_surplus_deficit.png", dpi=500)
plt.close()
print("Chart 6 saved.")

Chart 6 saved.


In [32]:
print("All charts generated and saved successfully.")

All charts generated and saved successfully.
